# Dirichlet Problem Solver

Numerical solution of the Dirichlet problem via finite differences and Poisson integral.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Finite difference solver for Dirichlet problem on unit disc
def solve_dirichlet_fd(boundary_func, grid_size=50, max_iter=5000, tol=1e-6):
    N = grid_size
    u = np.zeros((N+1, N+1))

    # Set boundary conditions
    for i in range(N+1):
        for j in range(N+1):
            x = -1 + 2*i/N
            y = -1 + 2*j/N
            r = np.sqrt(x**2 + y**2)
            if r >= 0.99:
                theta = np.arctan2(y, x)
                u[i, j] = boundary_func(theta)

    # Jacobi iteration
    h = 2.0 / N
    for _ in range(max_iter):
        u_old = u.copy()
        for i in range(1, N):
            for j in range(1, N):
                x = -1 + 2*i/N
                y = -1 + 2*j/N
                r = np.sqrt(x**2 + y**2)
                if r < 0.99:
                    u[i, j] = 0.25 * (u_old[i+1, j] + u_old[i-1, j] +
                                      u_old[i, j+1] + u_old[i, j-1])
        if np.max(np.abs(u - u_old)) < tol:
            break
    return u

print("Dirichlet FD solver defined")

In [ ]:
# Solve with boundary f(theta) = sin(theta)
boundary = lambda t: np.sin(t)
u = solve_dirichlet_fd(boundary, grid_size=30, max_iter=2000)

# Check at origin: should be 0 (average of sin over circle)
mid = u.shape[0] // 2
print(f"u(0,0) = {u[mid, mid]:.6f} (expected: ~0)")

# Check maximum
print(f"max u = {u.max():.4f}, min u = {u.min():.4f}")

In [ ]:
# Compare with Poisson integral solution
def poisson_integral_numeric(boundary, r, theta, N=200):
    dt = 2*np.pi/N
    result = 0.0
    for k in range(N):
        t = k*dt
        P = (1 - r**2) / (1 - 2*r*np.cos(theta - t) + r**2)
        result += P * boundary(t) * dt
    return result / (2*np.pi)

for r in [0.1, 0.3, 0.6, 0.9]:
    pi_val = poisson_integral_numeric(boundary, r, 0.0)
    print(f"r={r:.1f}, Poisson integral: u(r,0) = {pi_val:.6f}")